# 02 — TransferGuard: score, adapt, and fall back

This notebook proposes a **validation-based** way to avoid a harmful transfer choice. The source score and decay warning are exploratory. The operational decision is the existing P5 paired guard: choose the best adaptation on selection images, then transfer only if its paired bootstrap lower bound over scratch is above zero on the untouched guard images.

Run this notebook after `01_negative_transfer.ipynb`. No training or final test evaluation runs by default. New artifacts go under `runs/notebooks/transferguard`; P1–P5 artifacts are unchanged.

## Protocol

Use the same MobileNetV3-Small and the same 2,000 target training, 2,500 selection, and 2,500 guard indices as notebook 01. Frozen feature suitability is estimated from **training images only**. Source ranking precedes target adaptation. Scratch, frozen, late-layer, and full fine-tuning get the same target epoch budget. Selection accuracy chooses each checkpoint and the adaptation policy. The guard is read once for the final scratch fallback. The official test partition stays sealed until this entire choice is fixed.

For frozen feature matrix \(X\in\mathbb{R}^{N\times D}\), LogME treats each class indicator \(y_c\) as a Gaussian linear-regression target. It integrates weights \(w\) out of \(p(y_c\mid X,w,\beta)p(w\mid\alpha)\), optimizes the prior and noise precisions by fixed-point updates, divides log evidence by \(N\), then averages across classes. Higher LogME is a proxy for feature suitability, not an accuracy guarantee. The repository's verified float64 SVD implementation is used directly.

The paired guard bootstraps per-image correctness differences on the untouched guard set. Transfer is accepted only if the lower endpoint of its prespecified 95% interval exceeds zero; otherwise scratch is returned.

In [ ]:
from pathlib import Path
from copy import deepcopy
from datetime import datetime
from uuid import uuid4
import csv, json, hashlib, time, platform
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
import matplotlib.pyplot as plt

from experiments.baselines.train import seed_everything
from experiments.negative_transfer.train_target import load_svhn_model
from experiments.transferguard.run import target_loaders, train_model, predict_checkpoint
from src.datasets.cifar10 import get_cifar10_eval_transform
from src.guard.decision import paired_guard
from src.guard.policies import POLICIES, apply_policy
from src.models.mobilenet import get_model
from src.transferability.features import (target_training_subset, target_loader,
                                           load_source_model, extract_features)
from src.transferability.logme import logme_score

SEED, BATCH_SIZE, EPOCHS = 42, 16, 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SOURCE_CHECKPOINTS = {"svhn": None}  # Path to the P3 source best_model.pth, if available
P1_RUNS = {}                         # Explicit mapping source -> notebook 01 run path
RUN_LOGME = False
RUN_GUARD = False
EVALUATE_FINAL_TEST = False
LR = {"scratch": 1e-3, "frozen": 1e-3, "late": 1e-4, "full": 1e-4}
BOOTSTRAP_SAMPLES = 10000
ROOT = Path("runs/notebooks/transferguard")
seed_everything(SEED)
print("device:", DEVICE)

## Fixed split and feature extraction

`target_loaders` is the P5 split implementation. `target_training_subset` retrieves the identical training indices with deterministic evaluation transforms for frozen feature extraction. An explicit index assertion catches a mismatch before scoring. Features are average pooled from MobileNetV3-Small `features[12]` (576 dimensions); their extraction order and labels are fixed.

In [ ]:
train_loader, selection_loader, guard_loader, identity = target_loaders(BATCH_SIZE, SEED)
feature_subset, feature_identity = target_training_subset(train_size=2000, val_size=5000,
                                                          seed=SEED)
assert feature_identity["indices_sha256"] == identity["indices_sha256"]
assert feature_identity["labels_sha256"] == identity["labels_sha256"]
feature_loader = target_loader(feature_subset, BATCH_SIZE)
print({k: len(loader.dataset) for k, loader in
       [("train", train_loader), ("selection", selection_loader), ("guard", guard_loader)]})

In [ ]:
def source_features(source, checkpoint=None):
    seed_everything(SEED)
    model, source_info = load_source_model(source, checkpoint)
    features, labels = extract_features(model.to(DEVICE), feature_loader, DEVICE, ("final",))
    return features["final"], labels, source_info

scores = {}
score_file = ROOT / f"logme_seed{SEED}.json"
if not RUN_LOGME and score_file.exists():
    saved = json.loads(score_file.read_text())
    if saved["seed"] != SEED or saved["identity"] != identity:
        raise ValueError("Saved LogME scores use a different target split")
    scores = saved["scores"]
if RUN_LOGME:
    for source in ("imagenet", "svhn", "random"):
        checkpoint = SOURCE_CHECKPOINTS["svhn"] if source == "svhn" else None
        if source == "svhn" and checkpoint is None:
            continue
        start = time.perf_counter()
        x, labels, source_info = source_features(source, checkpoint)
        score = logme_score(x, labels)
        scores[source] = dict(score=score, seconds=time.perf_counter()-start,
                              feature_shape=list(x.shape), source_info=source_info)
    ROOT.mkdir(parents=True, exist_ok=True)
    if score_file.exists():
        raise FileExistsError(score_file)
    score_file.write_text(json.dumps(dict(seed=SEED, identity=identity, scores=scores),
                                     indent=2, default=str) + "\n")
    print({name: value["score"] for name, value in scores.items()})

## Transfer advantage over training

For each epoch \(e\), \(A_s(e)=100[a_s(e)-a_{scratch}(e)]\) percentage points on the same selection examples. A falling curve may warn that initial transfer benefit is decaying; it is **not** a calibrated detector. The fixed display statistic below is the mean advantage in the final third of epochs minus its mean in the first third. It is never fitted to guard or test outcomes. Guard fallback does not depend on this statistic.

In [ ]:
def accuracy_curve(run_dir):
    with (Path(run_dir) / "metrics.csv").open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    key = "selection_accuracy" if "selection_accuracy" in rows[0] else "val_accuracy"
    return np.asarray([float(row[key]) for row in rows])

def advantage_curve(source_run, scratch_run):
    source, scratch = accuracy_curve(source_run), accuracy_curve(scratch_run)
    if len(source) != len(scratch):
        raise ValueError("Epoch budgets differ")
    return 100 * (source - scratch)

if P1_RUNS:
    reference = json.loads((Path(P1_RUNS["scratch"])/"config.json").read_text())
    for source, path in P1_RUNS.items():
        candidate = json.loads((Path(path)/"config.json").read_text())
        for field in ("train_indices", "selection_indices", "guard_indices", "seed", "epochs"):
            if candidate[field] != reference[field]:
                raise ValueError(f"Incompatible {source} run: {field}")
        if source == "scratch":
            continue
        curve = advantage_curve(path, P1_RUNS["scratch"])
        third = max(1, len(curve)//3)
        decay = float(curve[-third:].mean()-curve[:third].mean())
        print(source, "early-to-late advantage change (pp):", round(decay, 3),
              "exploratory decay warning:", decay < 0)
        plt.plot(np.arange(1, len(curve)+1), curve, label=source)
    plt.axhline(0, color="black", linewidth=0.8)
    plt.xlabel("Target epoch"); plt.ylabel("Transfer advantage (pp)")
    plt.legend(); plt.show()

## Compare adaptation policies

A source must be chosen by a prespecified frozen-feature score or stated as a controlled source comparison. The three adaptation policies use the existing P5 freeze boundary: `frozen` trains only the classifier, `late` trains feature blocks 9–12 and the classifier, and `full` trains all parameters. Frozen BatchNorm running statistics stay fixed. Scratch is a fourth candidate. The P5 training loop writes per-epoch curves, best checkpoints, wall time, and peak CUDA memory. Set `RUN_GUARD=True` only when the full comparison is intended.

In [ ]:
def choose_source(scores):
    available = {name: row for name, row in scores.items()
                 if name in ("imagenet", "svhn")}
    if not available:
        raise ValueError("Compute or load compatible ImageNet/SVHN LogME scores first")
    return max(available, key=lambda name: available[name]["score"])

def fresh_source_model(source):
    seed_everything(SEED)
    if source == "imagenet":
        return get_model("finetune")
    if source == "svhn":
        checkpoint = SOURCE_CHECKPOINTS["svhn"]
        if checkpoint is None:
            raise ValueError("Set the scored SVHN checkpoint path")
        model, path, _ = load_svhn_model(checkpoint)
        expected = scores["svhn"]["source_info"]["sha256"]
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError("SVHN checkpoint changed since LogME scoring")
        return model
    raise ValueError(source)

def new_run_dir():
    path = ROOT / f"{datetime.now():%Y%m%d-%H%M%S}-{uuid4().hex[:8]}"
    path.mkdir(parents=True, exist_ok=False)
    return path

In [ ]:
GUARD_RUN = None
if RUN_GUARD:
    source = choose_source(scores)
    source_model = fresh_source_model(source)
    GUARD_RUN = new_run_dir()
    config = dict(seed=SEED, batch_size=BATCH_SIZE, epochs=EPOCHS, source=source,
                  source_checkpoint=str(SOURCE_CHECKPOINTS["svhn"]) if source == "svhn" else None,
                  logme_scores=scores, identity=identity, lr=LR,
                  bootstrap_samples=BOOTSTRAP_SAMPLES, device=DEVICE.type,
                  architecture="MobileNetV3-Small",
                  validation_use="selection for checkpoint/policy; guard once for fallback",
                  test_use="sealed", torch_version=torch.__version__,
                  python_version=platform.python_version())
    (GUARD_RUN / "config.json").write_text(json.dumps(config, indent=2, default=str)+"\n")
    results = {}
    seed_everything(SEED)
    scratch = get_model("scratch")
    results["scratch"] = train_model(scratch, "scratch", train_loader, selection_loader,
                                      DEVICE, EPOCHS, LR["scratch"], GUARD_RUN/"scratch", SEED)
    del scratch
    for policy in POLICIES:
        model = apply_policy(deepcopy(source_model), policy)
        results[policy] = train_model(model, policy, train_loader, selection_loader,
                                      DEVICE, EPOCHS, LR[policy], GUARD_RUN/policy, SEED)
        del model
    best_policy = max(POLICIES, key=lambda name: results[name]["selection_accuracy"])
    transfer_model = apply_policy(deepcopy(source_model), best_policy)
    labels, transfer_predictions = predict_checkpoint(
        transfer_model, GUARD_RUN/best_policy/"best_model.pth", guard_loader, DEVICE)
    scratch_model = get_model("scratch")
    scratch_labels, scratch_predictions = predict_checkpoint(
        scratch_model, GUARD_RUN/"scratch"/"best_model.pth", guard_loader, DEVICE)
    if labels != scratch_labels:
        raise RuntimeError("Guard label/order mismatch")
    guard = paired_guard(labels, transfer_predictions, scratch_predictions,
                         seed=SEED, bootstrap_samples=BOOTSTRAP_SAMPLES)
    summary = dict(source=source, selected_policy=best_policy, guard=guard,
                   final_decision=guard["decision"], training=results,
                   total_training_seconds=sum(r["training_seconds"] for r in results.values()))
    (GUARD_RUN/"summary.json").write_text(json.dumps(summary, indent=2)+"\n")
    print(summary)

## Ablations and visual checks

The saved candidate runs support simple comparisons without a second training pass: frozen vs late vs full, each against scratch; LogME ranking vs subsequent selection accuracy; and the chosen transfer policy with versus without the fixed guard fallback. These are **validation** comparisons. Report both accuracy and training cost. A guard decision is one estimate on one split, so repeat the entire protocol across prespecified seeds before general conclusions.

In [ ]:
if GUARD_RUN is not None:
    summary = json.loads((GUARD_RUN/"summary.json").read_text())
    rows = []
    for name, result in summary["training"].items():
        rows.append((name, result["selection_accuracy"], result["training_seconds"],
                     result["gpu_peak_bytes"]))
    print("policy | selection accuracy | training seconds | peak GPU bytes")
    for row in rows:
        print(*row, sep=" | ")
    plt.scatter([row[2] for row in rows], [row[1] for row in rows])
    for name, accuracy, seconds, _ in rows:
        plt.annotate(name, (seconds, accuracy))
    plt.xlabel("Target training seconds"); plt.ylabel("Selection accuracy"); plt.show()
    print("Guarded choice:", summary["final_decision"])
    print("Unguarded transfer choice:", summary["selected_policy"])
    print("Guard difference and 95% interval:", summary["guard"]["accuracy_difference"],
          (summary["guard"]["ci_lower"], summary["guard"]["ci_upper"]))

## Final test, once

Only after the source ranking, epoch budget, rates, adaptation policy, and guard rule have been fixed may the official CIFAR-10 test partition be evaluated. This optional cell uses the already selected checkpoint and writes `test_result.json` once. Do not use the test result to revise the warning or fallback rule. For a multi-seed study, run a fresh sealed protocol for every seed.

In [ ]:
if EVALUATE_FINAL_TEST:
    if GUARD_RUN is None:
        raise ValueError("Set GUARD_RUN to a completed new TransferGuard run")
    summary = json.loads((GUARD_RUN/"summary.json").read_text())
    result_file = GUARD_RUN/"test_result.json"
    if result_file.exists():
        raise FileExistsError("This run already has a final test result")
    test_data = CIFAR10("data/raw", train=False, download=True,
                        transform=get_cifar10_eval_transform())
    test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    if summary["final_decision"] == "scratch":
        chosen = "scratch"
        model = get_model("scratch")
    else:
        chosen = summary["selected_policy"]
        model = apply_policy(fresh_source_model(summary["source"]), chosen)
    labels, predictions = predict_checkpoint(model, GUARD_RUN/chosen/"best_model.pth",
                                              test_loader, DEVICE)
    accuracy = float(np.mean(np.asarray(labels) == np.asarray(predictions)))
    result_file.write_text(json.dumps(dict(chosen=chosen, test_accuracy=accuracy,
                                           test_size=len(labels)), indent=2)+"\n")
    print("Sealed test accuracy:", accuracy)

## Verification and migration status

The notebooks intentionally call the P1 balanced split, P2 epoch/checkpoint path, P3 source loader, P4 feature extractor and verified LogME, and P5 policy and guard functions. Before removing any module, run the existing unit tests and compare notebook outputs against the same indices, input tensors, and checkpoints. The repository keeps those modules until such parity has been executed. Source scoring and guard results from this notebook are new runs; no claim of novelty or performance improvement is made without repeated experiments.